# Train from the captured heat-gun run

This notebook uses `data/heatgun_box_001.csv` exactly as captured. It demonstrates a 20-second DHT window, a 60-second thermal-risk horizon, labels, scaling, and a tiny dense neural network: 20 inputs → 12 ReLU units → 3 Softmax outputs. It is a one-run teaching model, not a validated fire detector.

## Teaching labels

- `NORMAL`: DHT temperature remains below 35 C and does not approach the 50.7 C heat-gun boundary in the next minute.
- `ELEVATED_THERMAL_RISK`: warming trend, temperature at least 35 C, or reaches the known high boundary within 60 seconds.
- `HIGH_THERMAL_RISK`: current DHT temperature is at least 50.7 C.

These labels come from one controlled run. They show the method; they do not prove general fire prediction.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path('..').resolve()
sys.path.insert(0, str(project_root / 'tools'))
import train_heatgun_neural_net as training

data = pd.read_csv(project_root / training.SOURCE)
display(data.head())
print('Rows:', len(data))
print('Temperature range:', data.temp_c.min(), 'to', data.temp_c.max(), 'C')

In [ ]:
X, y = training.make_training_data(data)
counts = {training.LABELS[index]: int((y == index).sum()) for index in range(3)}
print('Input shape:', X.shape, '# 10 samples x 2 values = 20 features')
print('Class counts:', counts)
print('Forecast horizon:', training.FORECAST_HORIZON_SAMPLES * training.SAMPLE_SECONDS, 'seconds')

In [ ]:
import os
os.chdir(project_root)
training.main()
print('Model:', project_root / 'artifacts/heatgun_teaching_model.json')
print('Report:', project_root / 'artifacts/heatgun_teaching_model_report.json')

## Next teaching step

Read the exported model JSON: it contains normalization means/scales plus both dense-layer weights and biases. The next firmware step can implement this tiny neural network directly on ESP32. Later, reproduce the same structure in TensorFlow and export INT8 TFLite.